# Transformer Inference & Self-Attention

What happens inside a transformer forward pass: embeddings -> self-attention -> logits -> a prediction. We compute a real, from-scratch scaled dot-product attention over toy vectors with plain Python/NumPy (no model weights needed, no network), then show the real Hugging Face `pipeline()` call this replaces in a normal-network environment.

**Network note:** same sandbox constraint as the previous two notebooks — see 01_tokenization.ipynb.

## 1. Scaled dot-product self-attention, from scratch

Given query/key/value matrices Q, K, V for a short sequence, attention output is `softmax(Q Kᵀ / sqrt(d_k)) V` — each position produces a weighted average of every other position's value vector, weighted by how relevant their key is to its query.

In [1]:
import numpy as np

np.random.seed(0)
seq_len, d_k = 4, 8
tokens = ['payment', 'service', 'is', 'failing']

Q = np.random.randn(seq_len, d_k)
K = np.random.randn(seq_len, d_k)
V = np.random.randn(seq_len, d_k)

def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)

scores = Q @ K.T / np.sqrt(d_k)
attention_weights = softmax(scores)
output = attention_weights @ V

print('attention weights (rows sum to 1 - one row per query token):')
print(np.round(attention_weights, 2))
print()
print('output shape:', output.shape, '(same shape as input - one vector per token)')

attention weights (rows sum to 1 - one row per query token):
[[0.3  0.3  0.19 0.21]
 [0.26 0.38 0.24 0.12]
 [0.09 0.08 0.12 0.71]
 [0.82 0.09 0.02 0.07]]

output shape: (4, 8) (same shape as input - one vector per token)


In [2]:
print('Which tokens each token attends to most:')
for i, token in enumerate(tokens):
    top = np.argsort(-attention_weights[i])[:2]
    print(f'  {token!r:12} attends most to: {[tokens[j] for j in top]}')

Which tokens each token attends to most:
  'payment'    attends most to: ['service', 'payment']
  'service'    attends most to: ['service', 'payment']
  'is'         attends most to: ['failing', 'is']
  'failing'    attends most to: ['payment', 'service']


With random, untrained Q/K/V projections these attention weights are meaningless — a trained model learns projections where attention weights reflect real linguistic relationships (e.g. a verb attending strongly to its subject). The mechanism is identical; only the learned weights differ.

## 2. Real Hugging Face inference

This is the real code for a normal-network environment — a pretrained model doing real sentiment classification.

In [3]:
from transformers import pipeline

try:
    classifier = pipeline('sentiment-analysis', model='distilbert-base-uncased-finetuned-sst-2-english')
    result = classifier('Payment service is failing in production and customers cannot checkout.')
    print(result)
except Exception as exc:
    print('Could not reach huggingface.co from this sandbox (expected here):')
    print(f'  {type(exc).__name__}: {exc}')
    print()
    print("On a normal-network machine this prints something like:")
    print("  [{'label': 'NEGATIVE', 'score': 0.998...}]")

/home/user/ai-agent/agent-service/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


'(MaxRetryError("HTTPSConnectionPool(host='huggingface.co', port=443): Max retries exceeded with url: /distilbert-base-uncased-finetuned-sst-2-english/resolve/main/config.json (Caused by ProxyError('Unable to connect to proxy', OSError('Tunnel connection failed: 403 Forbidden')))"), '(Request ID: 7e3c04f5-148d-4491-b79d-1a67cc2852b4)')' thrown while requesting HEAD https://huggingface.co/distilbert-base-uncased-finetuned-sst-2-english/resolve/main/config.json


Retrying in 1s [Retry 1/5].


'(MaxRetryError("HTTPSConnectionPool(host='huggingface.co', port=443): Max retries exceeded with url: /distilbert-base-uncased-finetuned-sst-2-english/resolve/main/config.json (Caused by ProxyError('Unable to connect to proxy', OSError('Tunnel connection failed: 403 Forbidden')))"), '(Request ID: f4882669-b926-47fb-afd0-af76412053f2)')' thrown while requesting HEAD https://huggingface.co/distilbert-base-uncased-finetuned-sst-2-english/resolve/main/config.json


Retrying in 2s [Retry 2/5].


...

Retrying in 8s [Retry 4/5].


'(MaxRetryError("HTTPSConnectionPool(host='huggingface.co', port=443): Max retries exceeded with url: /distilbert-base-uncased-finetuned-sst-2-english/resolve/main/config.json (Caused by ProxyError('Unable to connect to proxy', OSError('Tunnel connection failed: 403 Forbidden')))"), '(Request ID: 79890613-91b4-43f7-999e-6bbf92d1914b)')' thrown while requesting HEAD https://huggingface.co/distilbert-base-uncased-finetuned-sst-2-english/resolve/main/config.json


Retrying in 8s [Retry 5/5].


'(MaxRetryError("HTTPSConnectionPool(host='huggingface.co', port=443): Max retries exceeded with url: /distilbert-base-uncased-finetuned-sst-2-english/resolve/main/config.json (Caused by ProxyError('Unable to connect to proxy', OSError('Tunnel connection failed: 403 Forbidden')))"), '(Request ID: 8aa731e8-7cda-4ee3-b8b2-fa6b04689d96)')' thrown while requesting HEAD https://huggingface.co/distilbert-base-uncased-finetuned-sst-2-english/resolve/main/config.json


Could not reach huggingface.co from this sandbox (expected here):
  ProxyError: (MaxRetryError("HTTPSConnectionPool(host='huggingface.co', port=443): Max retries exceeded with url: /distilbert-base-uncased-finetuned-sst-2-english/resolve/main/config.json (Caused by ProxyError('Unable to connect to proxy', OSError('Tunnel connection failed: 403 Forbidden')))"), '(Request ID: 8aa731e8-7cda-4ee3-b8b2-fa6b04689d96)')

On a normal-network machine this prints something like:
  [{'label': 'NEGATIVE', 'score': 0.998...}]
